In [4]:
import pandas as pd
df = pd.read_excel('skin_irritation.xlsx', sheet_name=2)

df.columns

df_selec = df[(df['Mixture']=='Chemical') & (df['Species']=='Human')] #Mixture 컬럼이 Chemical, Species 컬럼은 Human인 경우 선택.

df_selec['Endpoint'].value_counts()

df_class = df_selec[df_selec['Endpoint']=='Qualitative classification']

df_react = df_selec[df_selec['Endpoint']=='Positive reaction']

df_class['Response'].value_counts()

# Qualitatie classification 정리
df_class['label']=0  #label을 0으로 먼저 초기화하고,,,
df_class.loc[df_class['Response']!='Not classified','label']=1  #Not classified가 아니라면 (!=) label을 1로 설정.

# Positive reaction 정리
df_react['label']=0
df_react.loc[df_react['Response']!='0','label']=1 #Respons 값이 '0'이 아니라면 (!=) label을 1로 설정.

df_total = pd.concat([df_class, df_react])
df_total['label'].value_counts()


df_total['Chemical_Name']

df_total['Chemical_Name'].to_list()

chemical_name = df_total['Chemical_Name'].to_list()
unique_name = set(chemical_name) #set()을 사용하면, 중복된 item은 모두 삭제됩니다.

unique_name

len(unique_name) #180개 데이터가 있었지만, 단일 물질 기준으로는 119개가 있네요.




119

In [5]:
for each in unique_name:
    print (each)

#label 정보는 동일 화합물이라면 1이거나 0이어야 함. nunique()는 동일 화합물의 label에 있는 정보의 개수 확인.
#groupby(Chemical Name)은 화합물 이름을 기준으로 dataframe을 묶어봅니다.
#이름 기준으로 묶었을 때, label 정보가 몇개 있는지 확인하는 방식.
df_total.groupby('Chemical_Name')['label'].nunique()


df_total.groupby('Chemical_Name')['label'].nunique().max()

df_total.groupby('Chemical_Name')['label'].nunique().min()

label_counts = df_total.groupby('Chemical_Name')['label'].nunique()
bad_names = label_counts[label_counts > 1].index
print (bad_names)
print ('label 정보가 불일치하는 화합물 개수: ',len(bad_names))

df_filtered = df_total.groupby('Chemical_Name').filter(lambda x: x['label'].nunique() == 1)
print(f"전체: {df_total.shape}행")
print(f"label이 불일치 하는 화합물 제거: {df_filtered.shape}행")

df_cleaned = df_filtered.drop_duplicates(subset='Chemical_Name')
print(f"label이 일치 하는 화합물 중 중복된 결과 제거: {df_cleaned.shape}행")


#smiles 코드가 없는 물질?
df_cleaned['SMILES']

df_cleaned['SMILES'].isna()

df_smi = df_cleaned.dropna(subset=['SMILES'])

df_smi.shape

df_smi['label'].value_counts()

Fatty acids, potassium salts
Terpinyl acetate
Tetradecanoic acid
Alcohol ethoxylate C16-18/E5
4-(Methylthio)benzaldehyde
Polyethylene glycol
Sodium hydroxide
Sodium dodecyl sulfate
Alcohol ethoxylate C11/E3
Benzyl alcohol
1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)
Dimethylsulfoxide
Methyl laurate
Alkyl polyglucoside 600
Alcohol ethoxylate C11/E7
Citronellol
Ethylenediamine tetraacetic acid disodium salt
Dimethyl sulfoxide
Coco alkyltrimethylammonium chlorides
Ethylenediaminetetraacetic acid, disodium salt
1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one
Tris(hydroxymethyl)aminomethane
Benzyl salicylate
Ethanol
Butyl benzoate
alpha-Terpinyl acetate
Nonanoic acid
Sodium soap
Dipropylene glycol
Carbonic acid disodium salt, compd. with hydrogen peroxide (H2O2) (2:3)
1-Bromohexane
Alcohol ethoxylate C16-18/E14
1,2-Propylene glycol
Hexadecanoic acid
Methyl hexanoate
Hexadecanoic acid, 1-methylethyl ester
Tween 80
Octanol
Isopropyl tetradecanoate
Sodium dodecyl sulphate
Eugenol
Triet

label
0    56
1    25
Name: count, dtype: int64

In [6]:
no_smiles = df_cleaned[df_cleaned['SMILES'].isna()]['Chemical_Name'].unique()
print(no_smiles)

<StringArray>
[              'C12-13 beta-branched primary alco hol sulfate',
  'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
                       'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
                               'Alcohol ethoxylate C16-18/E14',
                                'Alcohol ethoxylate C16-18/E5',
                                'Alcohol ethoxylate C12-15/E3',
                                   'Alcohol ethoxylate C11/E7',
                                      'Alkyl dimethyl betaine',
                                       'Benzalkonium chloride',
                                              'Potassium soap',
                                                 'Sodium soap',
                                   'Hydrogenated tallow amine',
                                                  'Butan-1-ol',
                             'Cocotrimethyl ammonium chloride',
                          

In [15]:
# CAS 없는 데이터 확인
df_no_cas = df_total[df_total['CASRN'].isna()]
print(df_no_cas['Chemical_Name'].unique())

<StringArray>
[                     'Alcohol ethoxylate C12-15/E5 phosphate',
               'C12-13 beta-branched primary alco hol sulfate',
  'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
                       'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
                                     'Alkyl polyglucoside 600',
                               'Alcohol ethoxylate C16-18/E14',
                                'Alcohol ethoxylate C16-18/E5',
                                'Alcohol ethoxylate C12-15/E3',
                                   'Alcohol ethoxylate C11/E7',
                                   'Alcohol ethoxylate C11/E3']
Length: 11, dtype: str


In [16]:
print("총 CASRN 개수:", df_total['CASRN'].nunique())

총 CASRN 개수: 70


In [17]:
total_rows = df_total.shape[0]
print("전체 데이터 개수:", total_rows)

전체 데이터 개수: 180


In [18]:
unique_casrn = df_total['CASRN'].nunique()
print("고유 CASRN 개수:", unique_casrn)

고유 CASRN 개수: 70


In [19]:
duplicate_count = total_rows - unique_casrn
print("중복된 CASRN 개수:", duplicate_count)

중복된 CASRN 개수: 110


In [20]:
# CASRN별 label 종류 확인
label_counts = df_total.groupby('CASRN')['label'].nunique()

# 충돌 있는 CASRN만 확인
bad_cas = label_counts[label_counts > 1].index
print("label 충돌 있는 CASRN:", bad_cas)
print("충돌 개수:", len(bad_cas))

label 충돌 있는 CASRN: Index(['100-51-6', '106-24-1', '110-27-0', '111-27-3', '111-87-5', '112-30-1',
       '112-38-9', '112-39-0', '115-95-7', '143-07-7', '15120-21-5',
       '15630-89-4', '35296-72-1', '57-55-6', '629-19-6', '64-17-5', '64-19-7',
       '68424-94-2', '7647-01-0', '77-86-1', '7732-18-5', '9005-65-6',
       '97-53-0'],
      dtype='str', name='CASRN')
충돌 개수: 23


In [22]:
# label이 일관된 CASRN만 필터
df_filtered = df_total.groupby('CASRN').filter(lambda x: x['label'].nunique() == 1)

In [23]:
# CASRN 기준 중복 제거
df_cleaned = df_filtered.drop_duplicates(subset='CASRN')

In [24]:
df_final = df_cleaned.dropna(subset=['SMILES'])